Evaluate QR DQN Ablation Study 1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/MECE689_Bowling/MECE689_RL_Bowling_Atari
!ls -la

/content/drive/MyDrive/MECE689_Bowling/MECE689_RL_Bowling_Atari
total 41
drwx------ 2 root root  4096 Sep 25 15:17 code
drwx------ 2 root root  4096 Sep 25 15:04 .git
-rw------- 1 root root 13586 Oct 31 06:01 github_terminal.ipynb
-rw------- 1 root root    33 Sep 26 19:25 .gitignore
drwx------ 2 root root  4096 Sep 25 15:17 models
drwx------ 2 root root  4096 Oct 31 02:25 OLD
-rw------- 1 root root  2348 Sep 29 02:21 README.md
drwx------ 2 root root  4096 Sep 25 15:17 results
drwx------ 2 root root  4096 Oct 18 05:10 videos


In [ ]:
!pip install gymnasium[atari,accept-rom-license] ale-py sb3_contrib stable-baselines3

# Install tensorboard
!pip install tensorboard

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.2/187.2 kB 12.6 MB/s eta 0:00:00


In [ ]:
# Load tensorboard extension
%load_ext tensorboard

In [ ]:
# from stable_baselines3.common.evaluation import evaluate_policy
# evaluate_policy basically does what my manual for loop does
# evaluate_policy(model, env, n_eval_episodes=10, deterministic=True, render=False, warn=False)
# NOTE: I don't use this bc I need individual rewards for each episode for calculating HNS and HWRNS later

import os
import torch
import gymnasium as gym
import stable_baselines3
import ale_py
import numpy as np
import random

# Algo
from sb3_contrib import QRDQN

# For debugging
from stable_baselines3.common.monitor import Monitor
import time

# Action masking
from gymnasium import ActionWrapper
from stable_baselines3.common.atari_wrappers import AtariWrapper

# Vector environment
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack, DummyVecEnv

# Visualization
import moviepy.editor as mpy
from IPython.display import HTML
from base64 import b64encode
import matplotlib.pyplot as plt

print("All imports working")

In [ ]:
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
def convert(seconds):
    seconds = seconds % (24 * 3600)
    hour = seconds // 3600
    seconds %= 3600
    minutes = seconds // 60
    seconds %= 60

    return "%d:%02d:%02d" % (hour, minutes, seconds)

Visualize Training Curves

In [ ]:
# Activate Tensorboard
%tensorboard --logdir /content/drive/MyDrive/MECE689_Bowling/MECE689_RL_Bowling_Atari/code/qr_dqn_bowling_tensorboard/

Create Environment

In [ ]:
class ActionReducer(ActionWrapper):
  def __init__(self, env):
    super().__init__(env)

    # NOOP, FIRE, UP, and DOWN only. No UPFIRE. No DOWNFIRE.
    self.allowed_actions = [0,1,2,3]

    self.action_space = gym.spaces.Discrete(len(self.allowed_actions))

  def action(self, action):
    return self.allowed_actions[action]

In [ ]:
seed = 5000
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [ ]:
game_name = "ALE/Bowling-v5"

In [ ]:
n_envs = 1

# make_atari_env internally uses make_vec_env
# wrapper_kwargs passes clip_reward to the AtariWrapper
env = make_atari_env(
    game_name,
    n_envs=n_envs,         # Creates parallel envs that run simultaneously
    seed=seed,
    wrapper_kwargs=dict(clip_reward=False)
)

# n_stack gives 4 consecutive frames as input for each env
env = VecFrameStack(env, n_stack=4)

env.action_space.seed(seed)

Load Model

In [ ]:
# Current working directory:
# /content/drive/MyDrive/MECE689_Bowling/MECE689_RL_Bowling_Atari

# Load model
model_name = "qr_dqn_10000000_Ablation_Study_1"

model = QRDQN.load(
    f"models/{model_name}",
    env=env,
    device="cuda"
)

print("Model loaded")

Evaluate Model

In [ ]:
all_rewards = []
all_lengths = []
# total_episodes = 10
total_episodes = 10000   # 10K

print("Running evaluation of trained QR DQN agent")
start_time = time.time()

for episode in range(total_episodes):
    obs = env.reset()
    done = False
    total_reward = 0
    steps = 0

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, info = env.step(action)

        total_reward += reward[0]
        steps += 1

    # Record
    all_rewards.append(total_reward)
    all_lengths.append(steps)
    # print(f"Episode {episode+1}: Reward = {total_reward:6.1f}, Steps = {steps}")

# NOTE: this is faster than training bc no back prop is done, so no gradients need to be calculated
end_time = time.time()

# Calculate run time
training_duration = end_time - start_time
time_in_minutes_and_seconds = convert(training_duration)
print(f"Time taken: {time_in_minutes_and_seconds}")

SAVE episodes, reward, and steps to its own file

In [ ]:
results_folder = "/content/drive/MyDrive/MECE689_Bowling/MECE689_RL_Bowling_Atari/results"

all_rewards_path = f"{results_folder}/all_QR_dqn_Ablation_Study_1_rewards.npy"
all_lengths_path = f"{results_folder}/all_QR_dqn_Ablation_Study_1_lengths.npy"

In [ ]:
# Save to .npy files
np.save(all_rewards_path, np.array(all_rewards))
np.save(all_lengths_path, np.array(all_lengths))

print("Saved QR DQN Ablation Study 1 results to results folder!")

Video

In [ ]:
def evaluate_model_and_make_video(model, n_eval_episodes):
  """Evaluate model and return mean reward"""

  frames = []

  all_rewards = []
  all_steps = []
  for episode in range(n_eval_episodes):
    obs = env.reset()
    episode_reward = 0
    dones = [False]
    steps = 0

    while not dones[0]:
      # Predict next action
      action, _ = model.predict(obs, deterministic=True)
      obs, rewards, dones, infos = env.step(action)

      # Get a frame from the first environment inside the VecEnv
      frame = env.envs[0].render()
      frames.append(frame)

      if dones[0]:
        # Note: This check might not work as expected with a vectorized env (dones is an array)
        obs = env.reset()

      episode_reward += rewards[0]
      steps += 1
      # print(done)
      # Actions: NOOP(0), FIRE(1), UP(2), DOWN(3), UPFIRE(4), DOWNFIRE(5)
      # print(f"Reward earned for doing {action_dict[action[0]]}: {reward[0]}")

    print(f"Episode {episode+1}: Reward = {episode_reward:6.1f}, Steps = {steps}")
    all_rewards.append(episode_reward)
    all_steps.append(steps)

  # Save to mp4
  video_name = f"QR_dqn_Ablation_Study_1_eval.mp4"
  video_save_path = f"/content/drive/MyDrive/MECE689_Bowling/MECE689_RL_Bowling_Atari/videos/{video_name}"
  clip = mpy.ImageSequenceClip(frames, fps=30)
  clip.write_videofile(video_save_path)

  return all_rewards, all_steps, video_save_path

In [ ]:
n_eval_episodes = 3
# n_eval_episodes = 10
# n_eval_episodes = 30
# n_eval_episodes = 100

all_rewards, all_steps, video_save_path = evaluate_model_and_make_video(model, n_eval_episodes)
mean_reward = np.mean(all_rewards)
mean_steps = np.mean(all_steps)
print(f"{mean_reward:.4f} mean reward, {mean_steps:.4f} mean steps")

In [ ]:
# Display the video inline as part of Google Colab
mp4 = open(video_save_path, "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f'<video width=480 controls><source src="{data_url}" type="video/mp4"></video>')

In [ ]:
env.close()
del env

Load

In [ ]:
results_folder = "/content/drive/MyDrive/MECE689_Bowling/MECE689_RL_Bowling_Atari/results"

all_random_rewards_path = f"{results_folder}/all_random_rewards.npy"
all_random_lengths_path = f"{results_folder}/all_random_lengths.npy"

In [ ]:
# Load from .npy files
random_rewards_array = np.load(all_random_rewards_path)
random_lengths_array = np.load(all_random_lengths_path)

print("Loaded saved random agent results!")

In [ ]:
# Load from .npy files
rewards_array = np.load(all_rewards_path)
lengths_array = np.load(all_lengths_path)

print("Loaded saved QR DQN Ablation Study 1 agent results!")

Random Agent:

In [ ]:
print("Evaluation metrics of RANDOM AGENT:")
print(f"Total episodes: {total_episodes}")

print(f"Mean reward: {np.mean(random_rewards_array):.2f}")
print(f"Median reward: {np.median(random_rewards_array):.2f}")

print(f"Min reward: {np.min(random_rewards_array):.2f}")
print(f"Max reward: {np.max(random_rewards_array):.2f}")

print(f"Standard deviation: {np.std(random_rewards_array):.2f}")

print(f"Average episode length: {np.mean(random_lengths_array):.1f} steps")

QR DQN Ablation Study 1 Baseline:

In [ ]:
print("Evaluation metrics of trained QR DQN Ablation Study 1:")
print(f"Total episodes: {total_episodes}")

print(f"Mean reward: {np.mean(rewards_array):.2f}")
print(f"Median reward: {np.median(rewards_array):.2f}")

print(f"Min reward: {np.min(rewards_array):.2f}")
print(f"Max reward: {np.max(rewards_array):.2f}")

print(f"Standard deviation: {np.std(rewards_array):.2f}")

print(f"Average episode length: {np.mean(lengths_array):.1f} steps")

Histograms:

In [ ]:
# Plot histograms
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(rewards_array, alpha=0.7, label='Trained', bins=20)
plt.hist(random_rewards_array, alpha=0.7, label='Random', bins=20)
plt.xlabel('Episode Reward')
plt.ylabel('Frequency')
plt.legend()
plt.title('Reward Distribution: Trained vs Random')

# Plot box plot
plt.subplot(1, 2, 2)
plt.boxplot([rewards_array, random_rewards_array], labels=['Trained', 'Random'])
plt.ylabel('Episode Reward')
plt.title('Reward Comparison')

plt.tight_layout()
plt.show()

HNS and HWRNS

In [ ]:
def calculate_hns(agent_score, random_score, human_score):
  # Calculate Human Normalized Score (HNS)
  num = agent_score - random_score
  denom = human_score - random_score
  hns = num / denom
  return hns

In [ ]:
def calculate_hwrns(agent_score, random_score, world_record_score):
  # Calculate Human World Record Normalized Score (HWRNS)
  num = agent_score - random_score
  denom = world_record_score - random_score
  hwrns = num / denom
  return hwrns

In [ ]:
# HARD CODED VALUES:
# average human baseline
human_score = 161.0
# Max score is 300 via Perfect Game aka 12 strikes in a row
world_record_score = 300

In [ ]:
all_hns_values = []
all_hwrns_values = []

for i in range(total_episodes):
  agent_score = rewards_array[i]
  random_score = random_rewards_array[i]

  hns = calculate_hns(agent_score, random_score, human_score)
  # print(f"Episode {i+1}, HNS:   {(hns):.4f}")
  all_hns_values.append(hns)

  hwrns = calculate_hwrns(agent_score, random_score, world_record_score)
  # print(f"Episode {i+1}, HWRNS: {(hwrns):.4f}\n")
  all_hwrns_values.append(hwrns)

In [ ]:
# SAVE to files
all_hns_vals_array = np.array(all_hns_values)
all_hwrns_vals_array = np.array(all_hwrns_values)

hns_filename = "QR_dqn_Ablation_Study_1_hns_vals.npy"
hwrns_filename = "QR_dqn_Ablation_Study_1_hwrns_vals.npy"

np.save(f"{results_folder}/{hns_filename}", all_hns_vals_array)
np.save(f"{results_folder}/{hwrns_filename}", all_hwrns_vals_array)

In [ ]:
# Load files
all_hns_vals_array = np.load(f"{results_folder}/{hns_filename}")
all_hwrns_vals_array = np.load(f"{results_folder}/{hwrns_filename}")

In [ ]:
print("Evaluating HNS values:")
print(f"Mean HNS: {np.mean(all_hns_vals_array):.4f}")
print(f"Median HNS: {np.median(all_hns_vals_array):.4f}")

print(f"Min HNS: {np.min(all_hns_vals_array):.4f}")
print(f"Max HNS: {np.max(all_hns_vals_array):.4f}")

print(f"Standard deviation: {np.std(all_hns_vals_array):.4f}")

In [ ]:
print("Evaluating HWRNS values:")
print(f"Mean HWRNS: {np.mean(all_hwrns_vals_array):.4f}")
print(f"Median HWRNS: {np.median(all_hwrns_vals_array):.4f}")

print(f"Min HWRNS: {np.min(all_hwrns_vals_array):.4f}")
print(f"Max HWRNS: {np.max(all_hwrns_vals_array):.4f}")

print(f"Standard deviation: {np.std(all_hwrns_vals_array):.4f}")